This notebook runs the Exhaustive Grid Search for Cat/Dog Binary Classification

Pipeline flow:

```text
Sweet Spot cleaning config
→ image filtering
→ train/validation/test split
→ preprocessing search
→ frozen CNN feature extraction search
→ classical classifier search
→ select best validation config
→ evaluate selected config on held-out test set
```

## Completed exhaustive run summary

The completed run tested all **132 experiments** successfully:

```text
4 preprocessing modes × 1 image size × 3 backbones × 11 classifier variants = 132 experiments
```

Best validation configuration:

| Component | Selected value |
|---|---|
| Preprocessing | `augmented`, image size `224` |
| Feature extraction | `efficientnet_b0`, pretrained ImageNet |
| Classifier | `voting_soft` |
| Validation macro F1 | `0.997009` |
| Validation wrong predictions | `3 / 1003` |
| Test macro F1 | `0.993021` |
| Test wrong predictions | `7 / 1003` |

In [ ]:
from pathlib import Path
import os
import sys
import shutil

# Use one stable CUDA device on Colab.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# TODO: replace this with your real GitHub repository URL.
GITHUB_REPO_URL = "https://github.com/kahn-29/252-MachineLearning-Assignment1.git"
GITHUB_BRANCH = "main"

PROJECT_ROOT = Path("/content")
REPO_DIR = PROJECT_ROOT / "project_repo"

if "YOUR_USERNAME" in GITHUB_REPO_URL or "YOUR_REPOSITORY" in GITHUB_REPO_URL:
    raise ValueError("Please set GITHUB_REPO_URL to your real GitHub repository URL first.")

!pip install -q kagglehub imagehash opencv-python-headless

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 --branch "{GITHUB_BRANCH}" "{GITHUB_REPO_URL}" "{REPO_DIR}"

MODULES_DIR = REPO_DIR / "modules"
if not MODULES_DIR.exists():
    raise FileNotFoundError(f"Cannot find modules folder at: {MODULES_DIR}")

# Use modules directly from cloned repo.
# No /content/modules folder is created.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print("Using modules directly from:", MODULES_DIR)
for path in sorted(MODULES_DIR.glob("*.py")):
    print("-", path.name)

## 1. Import libraries and project modules

In [ ]:
from pathlib import Path
import copy
import json
import re
import time
import gc
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from modules.config_utils import (
    get_default_config,
    deep_update,
    validate_config,
    set_seed,
    get_device,
    ensure_dirs,
)

from modules.data_utils import (
    resolve_dataset_root,
    build_raw_dataframe,
    stratified_split,
    summarize_class_distribution,
    summarize_split_distribution,
)

from modules.image_audit import (
    audit_dataframe,
    describe_audit_metrics,
)

from modules.cleaning import (
    apply_cleaning,
    summarize_cleaning,
    summarize_removal_reasons,
    evaluate_cleaning_retention,
)

from modules.transforms import get_hybrid_transform
from modules.feature_extraction import extract_feature_splits
from modules.classical_models import train_classifier

from modules.evaluation import (
    evaluate_estimator,
    evaluate_predictions,
    find_wrong_predictions,
)

from modules.artifacts import (
    save_json,
    save_dataframe,
    save_pickle,
)

from modules.visualization import (
    plot_class_distribution_bar,
    plot_split_distribution,
    plot_grid_search_results,
    plot_confusion_matrix,
    plot_image_grid_from_df,
)

warnings.filterwarnings("ignore")

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

## 2. Experiment configuration

The cleaning block is the final result of the Sweet Spot experiment:

```python
{
    "enabled": False,
    "remove_corrupted": True,
    "remove_duplicates": True,
    "duplicate_hamming_threshold": 4,
}
```

Interpretation: the project does not apply aggressive quality-threshold filtering, but still keeps safety checks for corrupted images and near-duplicates.

In [ ]:
USER_CONFIG = {
    "paths": {
        "workspace": "/content/cat-dog-image-classifier",
        "data_dir": "/content/cat-dog-image-classifier/data",
        "raw_data_dir": "/content/cat-dog-image-classifier/data/raw",
        "processed_data_dir": "/content/cat-dog-image-classifier/data/processed",
        "features_dir": "/content/cat-dog-image-classifier/features",
        "models_dir": "/content/cat-dog-image-classifier/models",
        "reports_dir": "/content/cat-dog-image-classifier/reports",
        "figures_dir": "/content/cat-dog-image-classifier/reports/figures",
        "results_dir": "/content/cat-dog-image-classifier/reports/results",
    },
    "runtime": {
        "deterministic": False,
        "device": "auto",
        "num_workers": 2,
    },
    "dataset": {
        "dataset_id": "tongpython/cat-and-dog",
        "local_root": None,
        "kaggle_input_dir": "/content",
        "drop_unknown": True,
    },
    "cleaning": {
        "enabled": False,
        "remove_corrupted": True,
        "remove_duplicates": True,
        "duplicate_hamming_threshold": 4,
    },
    "split": {
        "train_ratio": 0.80,
        "val_ratio": 0.10,
        "test_ratio": 0.10,
        "label_col": "label",
    },
    "preprocessing": {
        "mode": "letterbox",
        "image_size": 224,
        "train_augmentation": False,
        "normalize": "imagenet",
    },
    "feature_extraction": {
        "backbone": "efficientnet_b0",
        "pretrained": True,
        "batch_size": 128,
        "num_workers": 2,
        "data_parallel": False,
        "force_recompute": False,
        "file_format": "npy",
        "on_error": "raise",
    },
    "classifier": {
        "name": "logistic_regression",
        "params": {
            "C": 1.0,
            "max_iter": 3000,
            "class_weight": None,
        },
    },
    "exhaustive_grid": {
        "primary_metric": "f1_macro",
        "tie_breakers": ["accuracy", "fit_time_seconds"],
        "preprocessing_modes": ["stretch", "center_crop", "letterbox", "augmented"],
        "image_sizes": [224],
        "backbones": ["efficientnet_b0", "resnet18", "vgg16"],
        "run_full_ensembles": True,
        "fail_fast": True,
    },
}

config = deep_update(get_default_config(), USER_CONFIG)
validate_config(config)

set_seed(config["seed"], deterministic=config["runtime"].get("deterministic", False))

workspace = Path(config["paths"]["workspace"])
reports_dir = Path(config["paths"]["results_dir"]) / "grid_search_exhaustive_132"
figures_dir = Path(config["paths"]["figures_dir"]) / "grid_search_exhaustive_132"
features_dir = Path(config["paths"]["features_dir"]) / "grid_search_exhaustive_132"
models_dir = Path(config["paths"]["models_dir"]) / "grid_search_exhaustive_132"

ensure_dirs(workspace, reports_dir, figures_dir, features_dir, models_dir)

device = get_device()

print("Device:", device)
print("Workspace:", workspace)
print("Reports:", reports_dir)
print("Figures:", figures_dir)
print("Features:", features_dir)
print("Models:", models_dir)

## 3. Orchestration helpers

These helpers keep the notebook readable while all reusable logic remains in the project modules.

In [ ]:
def safe_slug(value) -> str:
    text = str(value).lower().strip()
    text = re.sub(r"[^a-z0-9_.-]+", "-", text)
    return text.strip("-") or "value"


def set_by_dotted_key(cfg: dict, dotted_key: str, value):
    keys = str(dotted_key).split(".")
    cursor = cfg
    for key in keys[:-1]:
        if key not in cursor or not isinstance(cursor[key], dict):
            cursor[key] = {}
        cursor = cursor[key]
    cursor[keys[-1]] = copy.deepcopy(value)


def merge_overrides(base_cfg: dict, overrides: dict) -> dict:
    merged = copy.deepcopy(base_cfg)
    for key, value in overrides.items():
        if key in {"classifier_label", "experiment_type"}:
            continue
        if "." in str(key):
            set_by_dotted_key(merged, key, value)
        elif isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = {**merged[key], **copy.deepcopy(value)}
        else:
            merged[key] = copy.deepcopy(value)
    return merged


def feature_cache_key(exp_cfg: dict) -> str:
    return "__".join([
        f"mode-{safe_slug(exp_cfg['preprocessing']['mode'])}",
        f"size-{safe_slug(exp_cfg['preprocessing']['image_size'])}",
        f"norm-{safe_slug(exp_cfg['preprocessing'].get('normalize', 'imagenet'))}",
        f"backbone-{safe_slug(exp_cfg['feature_extraction']['backbone'])}",
        f"pretrained-{safe_slug(exp_cfg['feature_extraction'].get('pretrained', True))}",
    ])


def experiment_run_name(experiment: dict, index: int) -> str:
    parts = [
        f"mode-{safe_slug(experiment['preprocessing.mode'])}",
        f"size-{safe_slug(experiment['preprocessing.image_size'])}",
        f"backbone-{safe_slug(experiment['feature_extraction.backbone'])}",
        f"clf-{safe_slug(experiment['classifier_label'])}",
    ]
    return f"{index:03d}__" + "__".join(parts)


def build_feature_transforms(pre_cfg: dict):
    mode = pre_cfg["mode"]
    image_size = pre_cfg["image_size"]
    normalize = pre_cfg.get("normalize", "imagenet")

    # Augmentation is used only for training feature extraction.
    train_transform = get_hybrid_transform(
        mode=mode,
        image_size=image_size,
        train=(mode == "augmented"),
        normalize=normalize,
    )

    # Validation/test features must remain deterministic.
    eval_mode = "stretch" if mode == "augmented" else mode
    eval_transform = get_hybrid_transform(
        mode=eval_mode,
        image_size=image_size,
        train=False,
        normalize=normalize,
    )

    return train_transform, eval_transform


def sort_ranked_results(results_df: pd.DataFrame, primary_metric: str, tie_breakers=None) -> pd.DataFrame:
    ranked = results_df.copy()
    if "status" in ranked.columns:
        ranked = ranked[ranked["status"] == "ok"].copy()

    sort_cols = [primary_metric]
    for col in tie_breakers or []:
        if col in ranked.columns and col not in sort_cols:
            sort_cols.append(col)

    ascending = []
    for col in sort_cols:
        lowered = col.lower()
        ascending.append(any(token in lowered for token in ["time", "loss", "error", "duration"]))

    return ranked.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Build the 132-experiment grid

The grid contains:

```text
4 preprocessing modes
× 1 image size
× 3 backbones
× 11 classifier variants
= 132 experiments
```

The 11 classifier variants are parameterized versions of 5 classifier types.

In [ ]:
def build_classifier_grid():
    grid = []

    # Logistic Regression variants
    for C in [0.1, 1.0, 10.0]:
        grid.append({
            "classifier_label": f"LR_C{C}",
            "classifier.name": "logistic_regression",
            "classifier.params": {
                "C": C,
                "max_iter": 3000,
                "class_weight": None,
                "n_jobs": -1,
            },
            "experiment_type": "base",
        })

    # Linear SVM variants
    for C in [0.1, 1.0]:
        grid.append({
            "classifier_label": f"SVMLinear_C{C}",
            "classifier.name": "svm_linear",
            "classifier.params": {
                "C": C,
                "probability": True,
                "class_weight": None,
            },
            "experiment_type": "base",
        })

    # Random Forest variants
    for n_estimators in [100, 200]:
        for max_depth in [None, 20]:
            label_depth = "None" if max_depth is None else str(max_depth)
            grid.append({
                "classifier_label": f"RF_n{n_estimators}_depth{label_depth}",
                "classifier.name": "random_forest",
                "classifier.params": {
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "n_jobs": -1,
                    "class_weight": None,
                },
                "experiment_type": "base",
            })

    # Soft Voting ensemble
    grid.append({
        "classifier_label": "VotingSoft_default",
        "classifier.name": "voting_soft",
        "classifier.params": {
            "voting": "soft",
            "n_jobs": -1,
            "lr": {
                "C": 0.1,
                "max_iter": 3000,
                "n_jobs": -1,
            },
            "svm": {
                "C": 0.1,
                "probability": True,
            },
            "rf": {
                "n_estimators": 100,
                "n_jobs": -1,
            },
        },
        "experiment_type": "ensemble",
    })

    # Stacking ensemble
    grid.append({
        "classifier_label": "Stacking_default",
        "classifier.name": "stacking",
        "classifier.params": {
            "n_jobs": -1,
            "lr": {
                "C": 0.1,
                "max_iter": 3000,
                "n_jobs": -1,
            },
            "svm": {
                "C": 0.1,
                "probability": True,
            },
            "rf": {
                "n_estimators": 100,
                "n_jobs": -1,
            },
            "final_max_iter": 3000,
        },
        "experiment_type": "ensemble",
    })

    return grid


def build_experiment_grid(config: dict) -> list[dict]:
    grid_cfg = config["exhaustive_grid"]
    classifier_grid = build_classifier_grid()

    experiments = []
    for mode in grid_cfg["preprocessing_modes"]:
        for image_size in grid_cfg["image_sizes"]:
            for backbone in grid_cfg["backbones"]:
                for clf in classifier_grid:
                    if clf["experiment_type"] == "ensemble" and not grid_cfg.get("run_full_ensembles", True):
                        continue

                    experiments.append({
                        "preprocessing.mode": mode,
                        "preprocessing.image_size": image_size,
                        "feature_extraction.backbone": backbone,
                        "classifier.name": clf["classifier.name"],
                        "classifier.params": clf["classifier.params"],
                        "classifier_label": clf["classifier_label"],
                        "experiment_type": clf["experiment_type"],
                    })

    return experiments


experiments = build_experiment_grid(config)
experiment_grid_df = pd.DataFrame(experiments)

save_dataframe(experiment_grid_df, reports_dir / "experiment_grid_132.csv")

print("Number of experiments:", len(experiments))
display(experiment_grid_df.head(20))
display(
    experiment_grid_df
    .groupby(["preprocessing.mode", "feature_extraction.backbone"])
    .size()
    .reset_index(name="n_experiments")
)

## 5. Load dataset and inspect class distribution

The notebook downloads the public dataset using `kagglehub`.

Expected size from the final run: `10028` images.

In [ ]:
dataset_root = resolve_dataset_root(
    dataset_id=config["dataset"].get("dataset_id"),
    local_root=config["dataset"].get("local_root"),
    kaggle_input_dir=config["dataset"]["kaggle_input_dir"],
    extensions=config["dataset"]["extensions"],
    allow_download=True,
)

raw_df = build_raw_dataframe(
    root=dataset_root,
    extensions=config["dataset"]["extensions"],
    class_map=config["dataset"]["class_map"],
    drop_unknown=config["dataset"]["drop_unknown"],
)

raw_summary_df = summarize_class_distribution(raw_df, label_col="label_name")

save_dataframe(raw_df, reports_dir / "raw_image_index.csv")
save_dataframe(raw_summary_df, reports_dir / "raw_class_distribution.csv")

print("Dataset root:", dataset_root)
print("Raw dataframe shape:", raw_df.shape)

display(raw_summary_df)

plot_class_distribution_bar(
    raw_df,
    label_col="label_name",
    title="Raw Class Distribution",
    save_path=figures_dir / "raw_class_distribution.png",
    show=True,
)

## 6. Audit image quality

The audit step computes reusable quality metrics and perceptual hashes.

The output is cached as `audit_dataframe.csv`.

In [ ]:
audit_path = reports_dir / "audit_dataframe.csv"

if audit_path.exists():
    audit_df = pd.read_csv(audit_path)
else:
    audit_df = audit_dataframe(
        raw_df,
        path_col=config["audit"]["path_col"],
        label_col=config["audit"]["label_col"],
        label_name_col=config["audit"]["label_name_col"],
        compute_hash=config["audit"]["compute_hash"],
        preserve_cols=["sample_id"] if "sample_id" in raw_df.columns else None,
        show_progress=True,
    )
    save_dataframe(audit_df, audit_path)

audit_stats_df = describe_audit_metrics(audit_df)
save_dataframe(audit_stats_df, reports_dir / "audit_metric_summary.csv")

print("Audit dataframe shape:", audit_df.shape)
display(audit_stats_df.head(20))

## 7. Apply Sweet Spot cleaning

The final Sweet Spot config keeps valid images and avoids aggressive quality filtering.

In the completed run:

```text
Raw: 10028
Clean: 10028
Removed: 0
```

In [ ]:
clean_df, removed_df = apply_cleaning(audit_df, config["cleaning"])

cleaning_summary_df = summarize_cleaning(
    raw_df=audit_df,
    clean_df=clean_df,
    removed_df=removed_df,
    label_col="label_name",
)

removal_reasons_df = summarize_removal_reasons(
    removed_df,
    reason_col="removal_reasons" if "removal_reasons" in removed_df.columns else "removal_reason",
)

retention_report = evaluate_cleaning_retention(
    raw_df=audit_df,
    clean_df=clean_df,
    label_col="label",
)

save_dataframe(clean_df, reports_dir / "clean_dataframe.csv")
save_dataframe(removed_df, reports_dir / "removed_dataframe.csv")
save_dataframe(cleaning_summary_df, reports_dir / "cleaning_summary.csv")
save_dataframe(removal_reasons_df, reports_dir / "removal_reasons_summary.csv")
save_json(retention_report, reports_dir / "cleaning_retention_report.json")

print("Clean dataframe shape:", clean_df.shape)
print("Removed dataframe shape:", removed_df.shape)

display(cleaning_summary_df)
display(removal_reasons_df)
display(pd.DataFrame([retention_report]))

if not removed_df.empty:
    plot_image_grid_from_df(
        removed_df,
        n=12,
        path_col="path",
        title_col="label_name" if "label_name" in removed_df.columns else None,
        subtitle_cols=[
            col for col in ["removal_reason", "removal_reasons", "blur_laplacian", "min_side", "entropy"]
            if col in removed_df.columns
        ],
        random_sample=True,
        seed=config["seed"],
        suptitle="Removed Image Examples",
        save_path=figures_dir / "removed_image_examples.png",
        show=True,
    )
else:
    print("No images were removed by the selected Sweet Spot cleaning config.")

## 8. Train/validation/test split

The validation set selects the best grid-search configuration.  
The test set is used only once for final evaluation.

In [ ]:
train_df, val_df, test_df = stratified_split(
    clean_df,
    train_ratio=config["split"]["train_ratio"],
    val_ratio=config["split"]["val_ratio"],
    test_ratio=config["split"]["test_ratio"],
    seed=config["seed"],
    label_col=config["split"]["label_col"],
)

split_summary_df = summarize_split_distribution(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    label_col="label_name",
)

save_dataframe(train_df, reports_dir / "train_dataframe.csv")
save_dataframe(val_df, reports_dir / "val_dataframe.csv")
save_dataframe(test_df, reports_dir / "test_dataframe.csv")
save_dataframe(split_summary_df, reports_dir / "split_distribution.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(split_summary_df)

plot_split_distribution(
    splits={"train": train_df, "val": val_df, "test": test_df},
    label_col="label_name",
    save_path=figures_dir / "split_distribution.png",
    show=True,
)

## 9. One-experiment runner

Each experiment:

```text
build transform
→ extract/load features
→ train classifier
→ evaluate on validation set
```

Feature extraction is cached by preprocessing and backbone, so multiple classifiers can reuse the same feature matrices.

In [ ]:
def run_exhaustive_experiment(index: int, experiment: dict) -> dict:
    run_name = experiment_run_name(experiment, index)
    experiment_config = merge_overrides(config, experiment)

    pre_cfg = experiment_config["preprocessing"]
    feat_cfg = experiment_config["feature_extraction"]
    clf_cfg = experiment_config["classifier"]

    train_transform, eval_transform = build_feature_transforms(pre_cfg)
    feature_dir = features_dir / feature_cache_key(experiment_config)

    start_time = time.perf_counter()

    feature_result = extract_feature_splits(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        train_transform=train_transform,
        eval_transform=eval_transform,
        backbone_name=feat_cfg["backbone"],
        batch_size=feat_cfg.get("batch_size", 128),
        device=device,
        num_workers=feat_cfg.get("num_workers", 2),
        output_dir=feature_dir,
        pretrained=feat_cfg.get("pretrained", True),
        data_parallel=feat_cfg.get("data_parallel", False),
        force_recompute=feat_cfg.get("force_recompute", False),
        path_col="path",
        label_col="label",
        show_progress=True,
        on_error=feat_cfg.get("on_error", "raise"),
    )

    model = train_classifier(
        feature_result["X_train"],
        feature_result["y_train"],
        classifier_name=clf_cfg["name"],
        seed=experiment_config.get("seed", 42),
        **dict(clf_cfg.get("params", {})),
    )

    val_metrics = evaluate_estimator(
        model,
        feature_result["X_val"],
        feature_result["y_val"],
        model_name=run_name,
    )

    elapsed = time.perf_counter() - start_time

    record = {
        "run_name": run_name,
        "status": "ok",
        "experiment_type": experiment.get("experiment_type", "base"),
        "classifier_label": experiment.get("classifier_label", clf_cfg["name"]),
        "preprocessing.mode": pre_cfg["mode"],
        "preprocessing.image_size": pre_cfg["image_size"],
        "feature_extraction.backbone": feat_cfg["backbone"],
        "classifier.name": clf_cfg["name"],
        "classifier.params": json.dumps(clf_cfg.get("params", {}), default=str),
        "loaded_from_cache": bool(feature_result.get("loaded_from_cache", False)),
        "fit_time_seconds": round(float(elapsed), 6),
        "feature_dir": str(feature_dir),
        **val_metrics,
    }

    return {
        "record": record,
        "model": model,
        "feature_result": feature_result,
        "config": experiment_config,
    }

## 10. Run exhaustive grid search

The notebook saves intermediate results after every experiment.  
If Colab disconnects, rerunning can reuse cached feature files.

In [ ]:
results_path = reports_dir / "exhaustive_grid_132_results.csv"
intermediate_path = reports_dir / "exhaustive_grid_132_results_intermediate.csv"

records = []
failures = []
last_result = None

for index, experiment in enumerate(experiments):
    run_name = experiment_run_name(experiment, index)
    print(f"\n[{index + 1}/{len(experiments)}] {run_name}")

    try:
        last_result = run_exhaustive_experiment(index, experiment)
        records.append(last_result["record"])

    except Exception as exc:
        failure_record = {
            "run_name": run_name,
            "status": "failed",
            "error": repr(exc),
            **{k: v for k, v in experiment.items() if k != "classifier.params"},
            "classifier.params": json.dumps(experiment.get("classifier.params", {}), default=str),
        }
        records.append(failure_record)
        failures.append(failure_record)

        pd.DataFrame(records).to_csv(intermediate_path, index=False)

        if config["exhaustive_grid"].get("fail_fast", True):
            raise

    pd.DataFrame(records).to_csv(intermediate_path, index=False)
    clear_memory()

exhaustive_results_df = pd.DataFrame(records)
save_dataframe(exhaustive_results_df, results_path)

print("Finished experiments:", len(exhaustive_results_df))
print("Failures:", len(failures))

display(exhaustive_results_df.head())
display(exhaustive_results_df.tail())

## 11. Rank validation results

The best configuration is selected by validation macro F1.

Completed run best row:

```text
augmented + EfficientNet-B0 + Voting Soft
Validation macro F1 = 0.997009
Validation wrong predictions = 3 / 1003
```

In [ ]:
ranked_results_df = sort_ranked_results(
    exhaustive_results_df,
    primary_metric=config["exhaustive_grid"]["primary_metric"],
    tie_breakers=config["exhaustive_grid"].get("tie_breakers", ["accuracy"]),
)

save_dataframe(ranked_results_df, reports_dir / "exhaustive_grid_132_results_ranked.csv")

display(ranked_results_df.head(20))

## 12. Visualize grid-search results

These charts compare preprocessing, backbone, and classifier effects.

In [ ]:
plot_grid_search_results(
    ranked_results_df,
    x="feature_extraction.backbone",
    y=config["exhaustive_grid"]["primary_metric"],
    hue="classifier.name",
    save_path=figures_dir / "grid_by_backbone_classifier.png",
    show=True,
)

plot_grid_search_results(
    ranked_results_df,
    x="preprocessing.mode",
    y=config["exhaustive_grid"]["primary_metric"],
    hue="feature_extraction.backbone",
    save_path=figures_dir / "grid_by_preprocessing_backbone.png",
    show=True,
)

top20_df = ranked_results_df.head(20).copy()
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=top20_df, y="run_name", x=config["exhaustive_grid"]["primary_metric"], ax=ax)
ax.set_title("Top 20 Exhaustive Grid Experiments", fontweight="bold")
ax.set_xlabel(config["exhaustive_grid"]["primary_metric"])
ax.set_ylabel("")
ax.grid(axis="x", linestyle="--", alpha=0.35)
fig.tight_layout()
fig.savefig(figures_dir / "top20_exhaustive_grid.png", dpi=160, bbox_inches="tight")
plt.show()

## 13. Component-level summary and interpretation

From the completed run:

- `augmented` achieved the best maximum validation F1.
- `efficientnet_b0` clearly dominated the backbone comparison.
- `VotingSoft_default`, `Stacking_default`, and `LR_C0.1` reached the same top validation F1, but Voting Soft was selected as the best ranked row.

In [ ]:
component_summary = {
    "preprocessing": (
        ranked_results_df
        .groupby("preprocessing.mode")[config["exhaustive_grid"]["primary_metric"]]
        .agg(["count", "mean", "max"])
        .sort_values("max", ascending=False)
        .reset_index()
    ),
    "backbone": (
        ranked_results_df
        .groupby("feature_extraction.backbone")[config["exhaustive_grid"]["primary_metric"]]
        .agg(["count", "mean", "max"])
        .sort_values("max", ascending=False)
        .reset_index()
    ),
    "classifier": (
        ranked_results_df
        .groupby("classifier_label")[config["exhaustive_grid"]["primary_metric"]]
        .agg(["count", "mean", "max"])
        .sort_values("max", ascending=False)
        .reset_index()
    ),
    "classifier_type": (
        ranked_results_df
        .groupby("classifier.name")[config["exhaustive_grid"]["primary_metric"]]
        .agg(["count", "mean", "max"])
        .sort_values("max", ascending=False)
        .reset_index()
    ),
}

save_dataframe(component_summary["preprocessing"], reports_dir / "component_summary_preprocessing.csv")
save_dataframe(component_summary["backbone"], reports_dir / "component_summary_backbone.csv")
save_dataframe(component_summary["classifier"], reports_dir / "component_summary_classifier.csv")
save_dataframe(component_summary["classifier_type"], reports_dir / "component_summary_classifier_type.csv")

display(component_summary["preprocessing"])
display(component_summary["backbone"])
display(component_summary["classifier"])
display(component_summary["classifier_type"])

## 14. Reconstruct and export the best validation config

In [ ]:
best_row = ranked_results_df.iloc[0].to_dict()

best_experiment = None
best_index = None

for index, experiment in enumerate(experiments):
    if experiment_run_name(experiment, index) == best_row["run_name"]:
        best_experiment = experiment
        best_index = index
        break

if best_experiment is None:
    raise ValueError("Could not reconstruct best experiment from ranked results.")

best_grid_config = merge_overrides(config, best_experiment)

save_json(best_grid_config, reports_dir / "best_exhaustive_132_config.json")

print("Best validation row:")
print(json.dumps(best_row, indent=2, default=str))

print("\nBest config core:")
print(json.dumps({
    "cleaning": best_grid_config["cleaning"],
    "preprocessing": best_grid_config["preprocessing"],
    "feature_extraction": best_grid_config["feature_extraction"],
    "classifier": best_grid_config["classifier"],
}, indent=2, default=str))

## 15. Final evaluation on held-out test set

The validation set selects the model.  
The test set estimates final generalization performance.

Completed run result:

```text
Test macro F1 = 0.993021
Wrong predictions = 7 / 1003
ROC-AUC = 0.999849
```

In [ ]:
best_result = run_exhaustive_experiment(best_index, best_experiment)
best_model = best_result["model"]
best_features = best_result["feature_result"]

test_metrics = evaluate_estimator(
    best_model,
    best_features["X_test"],
    best_features["y_test"],
    model_name="best_exhaustive_132_config",
)

final_metrics_df = pd.DataFrame([
    {
        "split": "validation",
        **{
            k: best_row.get(k)
            for k in [
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "roc_auc",
                "wrong_predictions",
                "total_samples",
                "error_rate",
            ]
            if k in best_row
        },
    },
    {"split": "test", **test_metrics},
])

save_dataframe(final_metrics_df, reports_dir / "best_exhaustive_132_validation_test_metrics.csv")
save_pickle(best_model, models_dir / "best_exhaustive_132_model.pkl")

display(final_metrics_df)

## 16. Confusion matrix and wrong predictions

The wrong-prediction table supports qualitative analysis in the final report.

In [ ]:
test_pred = best_model.predict(best_features["X_test"])

test_prob = None
if hasattr(best_model, "predict_proba"):
    try:
        test_prob = best_model.predict_proba(best_features["X_test"])
    except Exception:
        test_prob = None

label_names = config["dataset"].get("label_names", {0: "cat", 1: "dog"})
label_map = {int(k): v for k, v in label_names.items()} if isinstance(label_names, dict) else {0: "cat", 1: "dog"}

evaluation_payload = evaluate_predictions(
    y_true=best_features["y_test"],
    y_pred=test_pred,
    y_prob=test_prob,
    labels=[0, 1],
    label_names=label_map,
)

test_report_df = evaluation_payload["report"]
test_cm_df = evaluation_payload["confusion_matrix"]
test_metric_dict = evaluation_payload["metrics"]

wrong_df = find_wrong_predictions(
    df=test_df,
    y_true=best_features["y_test"],
    y_pred=test_pred,
    y_prob=test_prob,
    path_col="path",
    label_map=label_map,
)

save_dataframe(test_report_df.reset_index().rename(columns={"index": "label"}), reports_dir / "best_exhaustive_132_test_classification_report.csv")
save_dataframe(test_cm_df, reports_dir / "best_exhaustive_132_test_confusion_matrix.csv")
save_dataframe(wrong_df, reports_dir / "best_exhaustive_132_test_wrong_predictions.csv")
save_json(test_metric_dict, reports_dir / "best_exhaustive_132_test_metrics.json")

display(pd.DataFrame([test_metric_dict]))
display(test_report_df)
display(test_cm_df)
display(wrong_df)

In [ ]:
plot_confusion_matrix(
    test_cm_df,
    title="Best Exhaustive 132 Config — Test Confusion Matrix",
    save_path=figures_dir / "best_exhaustive_132_test_confusion_matrix.png",
    show=True,
)

if not wrong_df.empty:
    plot_image_grid_from_df(
        wrong_df,
        n=12,
        path_col="path",
        title_col="true_label_name" if "true_label_name" in wrong_df.columns else None,
        subtitle_cols=[col for col in ["pred_label_name", "confidence"] if col in wrong_df.columns],
        suptitle="Best Exhaustive 132 Config — Wrong Predictions",
        save_path=figures_dir / "best_exhaustive_132_test_wrong_predictions.png",
        show=True,
    )
else:
    print("No wrong predictions on the test set.")

## 17. Export final report

The final report JSON contains dataset sizes, search axes, best validation row, selected config, and test metrics.

In [ ]:
final_grid_report = {
    "platform": "colab",
    "gpu_count": int(torch.cuda.device_count()),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
    "n_raw": int(len(raw_df)),
    "n_clean": int(len(clean_df)),
    "n_removed": int(len(removed_df)),
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "n_experiments": int(len(experiments)),
    "n_successful_experiments": int((exhaustive_results_df["status"] == "ok").sum()) if "status" in exhaustive_results_df.columns else int(len(exhaustive_results_df)),
    "n_failed_experiments": int((exhaustive_results_df["status"] == "failed").sum()) if "status" in exhaustive_results_df.columns else 0,
    "search_axes": {
        "preprocessing_modes": config["exhaustive_grid"]["preprocessing_modes"],
        "image_sizes": config["exhaustive_grid"]["image_sizes"],
        "backbones": config["exhaustive_grid"]["backbones"],
        "classifier_grid": build_classifier_grid(),
    },
    "best_validation_row": best_row,
    "best_config_core": {
        "cleaning": best_grid_config["cleaning"],
        "preprocessing": best_grid_config["preprocessing"],
        "feature_extraction": best_grid_config["feature_extraction"],
        "classifier": best_grid_config["classifier"],
    },
    "test_metrics": test_metric_dict,
    "wrong_predictions": int(len(wrong_df)),
    "outputs": {
        "reports_dir": str(reports_dir),
        "figures_dir": str(figures_dir),
        "features_dir": str(features_dir),
        "models_dir": str(models_dir),
    },
}

save_json(final_grid_report, reports_dir / "exhaustive_grid_132_final_report.json")

print(json.dumps(final_grid_report, indent=2, default=str))

# Final conclusion

The exhaustive grid search selected:

```text
Augmented 224
+ EfficientNet-B0 frozen feature extraction
+ Soft Voting classifier
```

as the best validation configuration.

The selected model achieved strong held-out test performance:

```text
Macro F1 ≈ 0.9930
Wrong predictions = 7 / 1003
```

## Final default config for the classical pipeline

```python
"cleaning": {
    "enabled": False,
    "remove_corrupted": True,
    "remove_duplicates": True,
    "duplicate_hamming_threshold": 4,
}

"preprocessing": {
    "mode": "augmented",
    "image_size": 224,
    "train_augmentation": False,
    "normalize": "imagenet",
}

"feature_extraction": {
    "backbone": "efficientnet_b0",
    "pretrained": True,
    "batch_size": 128,
    "num_workers": 2,
    "data_parallel": False,
    "force_recompute": False,
    "file_format": "npy",
    "on_error": "raise",
}

"classifier": {
    "name": "voting_soft",
    "params": {
        "voting": "soft",
        "n_jobs": -1,
        "lr": {
            "C": 0.1,
            "max_iter": 3000,
            "n_jobs": -1,
        },
        "svm": {
            "C": 0.1,
            "probability": True,
        },
        "rf": {
            "n_estimators": 100,
            "n_jobs": -1,
        },
    },
}
```